# Abstractive Text Summarization: fine-tuning T5-base on BBC News

B.Tech ECE final-year project, Techno India University (team of 4).

This notebook fine-tunes Google's pre-trained **T5-base** model on the **BBC News Summary** dataset (2,225 article/summary pairs) using Hugging Face Transformers.

**How to run (Google Colab):**
1. Open this notebook in Google Colab and choose **Runtime > Change runtime type > GPU**.
2. Download the **BBC News Summary** dataset from Kaggle as `archive.zip` and upload it to Colab (left sidebar, Files).
3. Run the cells from top to bottom.
4. At the end, the trained model is saved to Google Drive as `BTech_Project_Model`. Download that folder and place it next to `app.py`.

## 1. Install libraries

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

## 2. (Optional) Kaggle credentials
Only needed if you download the dataset with the Kaggle command line tool. If you uploaded `archive.zip` by hand, you can skip this cell. Never upload `kaggle.json` to GitHub.

In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## 3. Load the BBC News Summary dataset

In [ ]:
# Unzip the dataset archive into a new folder
!unzip -q "archive.zip" -d bbc_data

In [ ]:
import pandas as pd
import os

# Update this base_path if 'ls -R' showed the folders are deeper (e.g., 'bbc_data/BBC News Summary')
base_path = '/content/bbc_data/BBC News Summary'

def load_bbc_data(path):
    articles_dir = os.path.join(path, 'News Articles')
    summaries_dir = os.path.join(path, 'Summaries')
    categories = ['business', 'entertainment', 'politics', 'sport', 'tech']

    data = []
    for cat in categories:
        art_cat_path = os.path.join(articles_dir, cat)
        sum_cat_path = os.path.join(summaries_dir, cat)

        if not os.path.exists(art_cat_path):
            continue

        for filename in os.listdir(art_cat_path):
            if filename.endswith(".txt"):
                # Use latin1 encoding as the BBC dataset often has special characters
                with open(os.path.join(art_cat_path, filename), 'r', encoding='latin1') as f:
                    article = f.read().strip()
                with open(os.path.join(sum_cat_path, filename), 'r', encoding='latin1') as f:
                    summary = f.read().strip()

                data.append({
                    "input_text": "summarize: " + article,   # T5 task prefix
                    "target_text": summary
                })
    return pd.DataFrame(data)

df = load_bbc_data(base_path)
print(f"Dataset Loaded! Total pairs: {len(df)}")
df.head()

## 4. Tokenization and train/test split (90% / 10%)

In [ ]:
from datasets import Dataset
from transformers import T5Tokenizer

# 1. Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# 2. Load the T5 Tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-base")

def preprocess_function(examples):
    # 1. Tokenize the input articles
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # 2. Tokenize the target summaries using 'text_target'
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=150,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Split into Train and Test (90% training, 10% testing)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)

## 5. Fine-tune T5-base (3 epochs)

In [ ]:
from transformers import T5ForConditionalGeneration, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = T5ForConditionalGeneration.from_pretrained("t5-base")

# This handles padding for us automatically
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # Adjust based on GPU memory
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,              # 3 epochs is usually enough for this dataset
    predict_with_generate=True,
    fp16=True,                       # Mixed precision for Colab GPU speed
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

## 6. Test the model on a new article

In [ ]:
# 1. Define the sample news article (replace this with any news text)
new_article = """
At least 117 people died Wednesday across UP in lethal thunderstorms that left a trail of destruction, unleashing lightning strikes and cyclone-like winds of up to 130kmph — unheard of in recent memory in the state.
According to data released by the relief commissioner's office, 113 people died in the destruction wrought by the storms and collapsing structures, while four lost their lives in lightning strikes. The disaster, which unfolded Wednesday afternoon, also left 79 people injured, killed at least 177 animals and destroyed 330 houses across several districts. Crops such as mangoes suffered widespread damage.
The thundersqualls were triggered by a western disturbance colliding with localised cyclonic circulations. CM Yogi Adityanath announced compensation of Rs 4 lakh each for the families of the deceased.
The worst-affected district was Prayagraj, where 23 people died after strong winds uprooted trees, knocked down walls and damaged houses. Mirzapur followed with 19 deaths. Sant Ravidas Nagar recorded 16 casualties and Fatehpur 11. Other districts, including Rae Bareli, Unnao and Budaun, also witnessed significant loss of life.
India Meteorological Department confirmed that wind speeds neared cyclonic levels in several districts. Such winds swept up loose topsoil, turning the system into highly destructive dust storms accompanied by widespread lightning and hail.
"""

# 2. Prepare the input with the T5 prefix
input_text = "summarize: " + new_article
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to("cuda")

# 3. Generate the summary
summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=5,               # More beams = broader search for a good summary
    max_length=120,            # Enough room for 2-3 full sentences
    min_length=40,
    no_repeat_ngram_size=3,
    repetition_penalty=2.5,    # Strongly discourages repeating phrases
    length_penalty=1.0,        # Balanced length
    early_stopping=True
)

full_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Clean up the end so it does not stop mid-sentence
if not full_summary.endswith('.'):
    full_summary = full_summary[:full_summary.rfind('.')+1]

print("--- Cleaned Summary ---")
print(full_summary)

## 7. Save the model to Google Drive
This folder (`BTech_Project_Model`) is what `app.py` loads. It is large (close to 1 GB), so do **not** upload it to GitHub. Download it from Drive and place it next to `app.py`, or upload it to the Hugging Face Hub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Create a path inside your Google Drive
save_path = "/content/drive/MyDrive/BTech_Project_Model"

# Save the model and tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model successfully saved to: {save_path}")

In [ ]:
# Optional: zip the folder so it is easier to download
!zip -r /content/BTech_Model.zip /content/drive/MyDrive/BTech_Project_Model